# Course Difficulty Features

**Step A** — Load `df_train`, `df_valid`, `df_test` from the pre-built parquet files saved by `split_diagnostics.ipynb`. Run that notebook first if the files don't exist.

**Step B** — Build per-course difficulty features from `df_train` only (empirical Bayes shrinkage, 3-level fallback chain, leave-one-out), join to all three splits, and save.

The split boundaries and exclusion logic live entirely in `split_diagnostics.ipynb`. This notebook never touches `df_primary` or redefines any year/semester masks.

## Step A — Load Pre-built Splits

The temporal split was defined and saved by `split_diagnostics.ipynb`. This notebook loads the three parquet files directly — it never redefines year/semester boundaries or touches `df_primary`.

If any sanity check below fails, re-run `split_diagnostics.ipynb` first.

**Step A.0 — Configuration.** `SPLIT_DATA_DIR` must match the same constant in `split_diagnostics.ipynb`. Edit this path here if the split files have moved.

In [1]:
SPLIT_DATA_DIR = 'D:/AI/Real projects/Academic_Advisor/data/model_data'

**Step A.1** — Load `df_train`, `df_valid`, `df_test` from the saved parquet files. Print each shape immediately to confirm the files loaded correctly.

In [2]:
import pandas as pd
import numpy as np
import os

TRAIN_PATH = os.path.join(SPLIT_DATA_DIR, 'df_train.parquet')
VALID_PATH = os.path.join(SPLIT_DATA_DIR, 'df_valid.parquet')
TEST_PATH  = os.path.join(SPLIT_DATA_DIR, 'df_test.parquet')

df_train = pd.read_parquet(TRAIN_PATH)
df_valid = pd.read_parquet(VALID_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f'df_train : {df_train.shape}')
print(f'df_valid : {df_valid.shape}')
print(f'df_test  : {df_test.shape}')

df_train : (450465, 61)
df_valid : (156097, 61)
df_test  : (110008, 61)


**What this shows:** Shapes confirm all three files loaded. If a file is missing, `pd.read_parquet` raises `FileNotFoundError` — re-run `split_diagnostics.ipynb` first. The column count here is the **before-enrichment** baseline; the save cell at the end will assert it increased by exactly 5.

**Step A.2** — Confirm the loaded splits have the expected year boundaries (train 2005–2021, val 2022–2023, test includes up to 2025-semester-1).

In [3]:
rows = []
for name, df in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    n = len(df)
    rows.append({
        'split':     name,
        'row_count': n,
        'min_year':  df['part_year'].min(),
        'max_year':  df['part_year'].max(),
    })
print(pd.DataFrame(rows).to_string(index=False))

split  row_count  min_year  max_year
train     450465      2005      2021
valid     156097      2022      2023
 test     110008      2024      2025


**What this shows:** Min/max year per split confirms the correct temporal boundaries. If `min_year` for train is 2013 instead of 2005, the loaded files are stale — re-run `split_diagnostics.ipynb` with the corrected boundaries first. This cell does **not** re-derive the split; it only confirms what's already in the files.

**Step A.3** — Hard sanity checks: 2016-semester-4 rows must be in `df_train` (kept intentionally), and 2025-semester-2 rows must be absent from all three splits (excluded as incomplete). Raises a clear `AssertionError` if either check fails.

In [4]:
# Check 1: 2016-semester-4 must be present in train (intentionally kept)
year_f_tr = df_train['part_year'].astype('float64')
sem_f_tr  = df_train['part_semester'].astype('float64')
n_2016_s4 = int(((year_f_tr == 2016) & (sem_f_tr == 4)).sum())

# Check 2: 2025-semester-2 must be absent from all three splits (excluded as incomplete)
n_2025_s2 = sum(
    int(((df['part_year'].astype('float64') == 2025) &
         (df['part_semester'].astype('float64') == 2)).sum())
    for df in [df_train, df_valid, df_test]
)

print(f'df_train rows with part_year==2016 AND part_semester==4 : {n_2016_s4:,}')
print(f'Rows with part_year==2025 AND part_semester==2 (all splits) : {n_2025_s2:,}')

assert n_2016_s4 > 0, (
    'FAIL: No 2016-semester-4 rows in df_train. '
    'The split files are stale or were built with the wrong train boundary. '
    'Re-run split_diagnostics.ipynb first.'
)
assert n_2025_s2 == 0, (
    f'FAIL: Found {n_2025_s2:,} rows with part_year==2025 AND part_semester==2 in the splits. '
    'The incomplete semester should have been excluded. '
    'Re-run split_diagnostics.ipynb first.'
)

print('\nSanity checks PASSED.')
print('  -> 2016-semester-4 rows are present in train (kept intentionally).')
print('  -> 2025-semester-2 rows are absent from all splits (excluded as incomplete).')

df_train rows with part_year==2016 AND part_semester==4 : 5,166
Rows with part_year==2025 AND part_semester==2 (all splits) : 0

Sanity checks PASSED.
  -> 2016-semester-4 rows are present in train (kept intentionally).
  -> 2025-semester-2 rows are absent from all splits (excluded as incomplete).


**What this shows:** Two hard checks that the loaded files were built with the correct logic. If either assertion fires, the error message tells you exactly what's wrong and instructs you to re-run `split_diagnostics.ipynb` — this notebook will not silently continue on stale or incorrect split files.

---

## Step B — Course Difficulty Features (train-only aggregation)

All aggregations in this step use **`df_train` exclusively**. `df_valid` and `df_test` are never touched until the join in B.13.

Key design decisions:
- Empirical Bayes shrinkage (MIN_SUPPORT = 20) prevents low-support courses from carrying spurious extremes.
- A 3-level fallback chain (exact key → course_id alone → global average) handles courses that appear in val/test but never in train.
- Leave-one-out for train rows prevents a row's own grade from leaking into its own feature value.

**Step B.9** — Aggregate per `degree_course_key` from `df_train`: support count, pass rate, average mark, retake rate.

In [5]:
# All aggregations are from df_train ONLY — val and test never touched here
# NaN final_mark / attempt_number values are excluded from each lambda via .dropna()

key_stats = (
    df_train
    .groupby('degree_course_key', as_index=False)
    .agg(
        support_count=('final_mark', 'count'),
        course_pass_rate=(
            'final_mark',
            lambda x: (x.dropna() >= 50).mean() if x.notna().any() else float('nan')
        ),
        course_avg_mark=('final_mark', 'mean'),
        course_retake_rate=(
            'attempt_number',
            lambda x: (x.dropna() > 1).mean() if x.notna().any() else float('nan')
        ),
    )
)

print(f'Unique degree_course_key in train : {len(key_stats):,}')
print(f'\nsupport_count distribution:')
print(key_stats['support_count'].describe().to_string())
print(f'\nKeys with support_count < 20  : {(key_stats["support_count"] < 20).sum():,}')
print(f'Keys with support_count == 1  : {(key_stats["support_count"] == 1).sum():,}')
print(f'\nFirst 5 rows:')
print(key_stats.head(5).to_string(index=False))

Unique degree_course_key in train : 1,666

support_count distribution:
count        1666.0
mean     270.387155
std      527.026279
min             1.0
25%             5.0
50%            53.0
75%          212.75
max          3180.0

Keys with support_count < 20  : 601
Keys with support_count == 1  : 141

First 5 rows:
degree_course_key  support_count  course_pass_rate  course_avg_mark  course_retake_rate
  1.111__1015.111            192           0.90625        76.119792            0.067708
  1.111__1016.111            914          0.974836        79.818381            0.040481
  1.111__1017.111            299          0.926421        77.414716            0.086957
  1.111__1018.111            524          0.885496        66.753817            0.158397
  1.111__1019.111            938          0.934968        78.590618            0.085288


**What this shows:** How many distinct `degree_course_key` values exist in train, and how unevenly they're seen. A large number of keys with `support_count < 20` means shrinkage (B.10) will have a meaningful effect on a significant fraction of keys. Keys with `support_count == 1` will collapse almost entirely to the global average after shrinkage.

**Step B.10** — Apply empirical Bayes shrinkage to `course_pass_rate` and `course_avg_mark`. Formula: `(n * raw + MIN_SUPPORT * global) / (n + MIN_SUPPORT)`. Print a before/after comparison for 10 low-support and 10 high-support keys.

In [6]:
MIN_SUPPORT = 20  # course with exactly this many observations is blended 50/50 with global avg

global_pass_rate = (df_train['final_mark'].dropna() >= 50).mean()
global_avg_mark  = df_train['final_mark'].mean()

print(f'MIN_SUPPORT      : {MIN_SUPPORT}')
print(f'Global pass rate : {global_pass_rate:.4f}')
print(f'Global avg mark  : {global_avg_mark:.4f}')

# Preserve raw values for the before/after printout
key_stats['course_pass_rate_raw'] = key_stats['course_pass_rate'].copy()
key_stats['course_avg_mark_raw']  = key_stats['course_avg_mark'].copy()

# Shrinkage: pull toward global mean weighted by how little we've seen the course
key_stats['course_pass_rate'] = (
    (key_stats['support_count'] * key_stats['course_pass_rate'] + MIN_SUPPORT * global_pass_rate)
    / (key_stats['support_count'] + MIN_SUPPORT)
)
key_stats['course_avg_mark'] = (
    (key_stats['support_count'] * key_stats['course_avg_mark'] + MIN_SUPPORT * global_avg_mark)
    / (key_stats['support_count'] + MIN_SUPPORT)
)

show_cols = ['degree_course_key', 'support_count',
             'course_pass_rate_raw', 'course_pass_rate',
             'course_avg_mark_raw',  'course_avg_mark']

pd.set_option('display.float_format', '{:.4f}'.format)

print(f'\n--- 10 LOWEST-support courses (shrinkage pulls strongly toward global) ---')
print(key_stats.nsmallest(10, 'support_count')[show_cols].to_string(index=False))

print(f'\n--- 10 HIGHEST-support courses (shrinkage has minimal effect) ---')
print(key_stats.nlargest(10, 'support_count')[show_cols].to_string(index=False))

pd.reset_option('display.float_format')

MIN_SUPPORT      : 20
Global pass rate : 0.8413
Global avg mark  : 65.5821

--- 10 LOWEST-support courses (shrinkage pulls strongly toward global) ---
degree_course_key  support_count  course_pass_rate_raw  course_pass_rate  course_avg_mark_raw  course_avg_mark
   1.111__211.111              1                0.0000            0.8013              17.0000          63.2687
   1.111__223.111              1                0.0000            0.8013              47.0000          64.6972
   1.111__224.111              1                0.0000            0.8013              30.0000          63.8877
   1.111__229.111              1                0.0000            0.8013              46.0000          64.6496
   1.111__230.111              1                1.0000            0.8489              60.0000          65.3163
   1.111__234.111              1                0.0000            0.8013              35.0000          64.1258
   1.111__237.111              1                0.0000            0.8013

**What this shows:** The shrinkage effect in action. Low-support courses (e.g., seen once) should show `course_pass_rate` much closer to the global baseline than `course_pass_rate_raw`. High-support courses should show very little movement. `course_retake_rate` is left unshrunk — it will still appear in the final output from B.9.

**Step B.11** — Build the level-2 (`course_id` alone, cross-degree) and level-3 (global train average) fallback structures. Also extract `course_id` from `degree_course_key` for use in the join.

In [7]:
# Extract course_id from degree_course_key (format: "{degree_id}__{course_id}")
key_stats['course_id'] = key_stats['degree_course_key'].str.rsplit('__', n=1).str[-1]

# Level-2: aggregate df_train by course_id alone (ignores which degree the course belongs to)
_cid_col = df_train['degree_course_key'].str.rsplit('__', n=1).str[-1]

cid_stats = (
    df_train
    .assign(course_id=_cid_col.values)
    .groupby('course_id', as_index=False)
    .agg(
        support_count=('final_mark', 'count'),
        course_pass_rate=(
            'final_mark',
            lambda x: (x.dropna() >= 50).mean() if x.notna().any() else float('nan')
        ),
        course_avg_mark=('final_mark', 'mean'),
        course_retake_rate=(
            'attempt_number',
            lambda x: (x.dropna() > 1).mean() if x.notna().any() else float('nan')
        ),
    )
)

# Apply the same shrinkage to level-2 stats
cid_stats['course_pass_rate'] = (
    (cid_stats['support_count'] * cid_stats['course_pass_rate'] + MIN_SUPPORT * global_pass_rate)
    / (cid_stats['support_count'] + MIN_SUPPORT)
)
cid_stats['course_avg_mark'] = (
    (cid_stats['support_count'] * cid_stats['course_avg_mark'] + MIN_SUPPORT * global_avg_mark)
    / (cid_stats['support_count'] + MIN_SUPPORT)
)

# Level-3: global averages (used when even course_id was never seen in train)
global_retake_rate = (df_train['attempt_number'].dropna() > 1).mean()
global_vals = {
    'course_pass_rate'  : global_pass_rate,
    'course_avg_mark'   : global_avg_mark,
    'course_retake_rate': global_retake_rate,
}

print(f'Level-1 (degree_course_key) unique keys : {len(key_stats):,}')
print(f'Level-2 (course_id alone)   unique keys : {len(cid_stats):,}')
print(f'Level-3 global averages:')
for k, v in global_vals.items():
    print(f'  {k:<28}: {v:.4f}')

Level-1 (degree_course_key) unique keys : 1,666
Level-2 (course_id alone)   unique keys : 811
Level-3 global averages:
  course_pass_rate            : 0.8413
  course_avg_mark             : 65.5821
  course_retake_rate          : 0.1606


**What this shows:** The level-2 key count is smaller than or equal to the level-1 count (many degree_course_keys collapse to the same course_id). The level-3 global values are the last-resort fallback. Val/test rows that never appeared in train at any level will receive these three numbers and `course_difficulty_fallback_level = 3`.

**Step B.12** — Leave-one-out (LOO) for train rows. Each train row's own outcome is subtracted from its course's running totals before shrinkage is applied, preventing self-leakage.

In [8]:
# --- Step 1: compute per-key totals from df_train ---
key_totals = (
    df_train
    .groupby('degree_course_key')
    .agg(
        n_fm   =('final_mark',     'count'),
        sum_pass=('final_mark',    lambda x: int((x.dropna() >= 50).sum())),
        sum_mark=('final_mark',    lambda x: float(x.dropna().sum())),
        n_at   =('attempt_number', 'count'),
        sum_ret=('attempt_number', lambda x: int((x.dropna() > 1).sum())),
    )
    .reset_index()
)

# --- Step 2: map totals onto every train row via .map() (preserves original index) ---
_idx = key_totals.set_index('degree_course_key')
df_train_loo = df_train.copy()
for col in ['n_fm', 'sum_pass', 'sum_mark', 'n_at', 'sum_ret']:
    df_train_loo[col] = df_train_loo['degree_course_key'].map(_idx[col])

# --- Step 3: this row's own contribution ---
_fm_ok  = df_train_loo['final_mark'].notna()
_this_pass  = ((df_train_loo['final_mark'] >= 50) & _fm_ok).astype(int)
_this_mark  = df_train_loo['final_mark'].fillna(0.0)
_this_fm_c  = _fm_ok.astype(int)

_at_ok  = df_train_loo['attempt_number'].notna()
_this_ret   = ((df_train_loo['attempt_number'] > 1) & _at_ok).astype(int)
_this_at_c  = _at_ok.astype(int)

# --- Step 4: LOO sums (subtract this row's contribution) ---
_n_loo         = (df_train_loo['n_fm']    - _this_fm_c).clip(lower=0)
_sum_pass_loo  = (df_train_loo['sum_pass']- _this_pass).clip(lower=0)
_sum_mark_loo  = (df_train_loo['sum_mark']- _this_mark)
_n_ret_loo     = (df_train_loo['n_at']    - _this_at_c).clip(lower=0)
_sum_ret_loo   = (df_train_loo['sum_ret'] - _this_ret).clip(lower=0)

# --- Step 5: shrunk LOO pass_rate and avg_mark ---
# When n_loo == 0 (singleton): denominator = MIN_SUPPORT, result = global average
df_train_loo['course_pass_rate'] = (
    (_sum_pass_loo + MIN_SUPPORT * global_pass_rate) / (_n_loo + MIN_SUPPORT)
)
df_train_loo['course_avg_mark'] = (
    (_sum_mark_loo + MIN_SUPPORT * global_avg_mark) / (_n_loo + MIN_SUPPORT)
)

# --- Step 6: unshrunk LOO retake rate (global fallback for singletons) ---
df_train_loo['course_retake_rate'] = np.where(
    _n_ret_loo.values > 0,
    (_sum_ret_loo / _n_ret_loo.replace(0, np.nan)).values,
    global_retake_rate
)

df_train_loo['course_difficulty_fallback_level'] = 1
df_train_loo['support_count'] = _n_loo.astype(int)

# Drop the intermediate aggregation columns before saving
_drop_cols = ['n_fm', 'sum_pass', 'sum_mark', 'n_at', 'sum_ret']
df_train_enriched = df_train_loo.drop(columns=_drop_cols)

n_singleton = int((_n_loo == 0).sum())
print(f'LOO enrichment complete: {len(df_train_enriched):,} train rows')
print(f'Singleton rows (n_loo == 0, collapse to global avg via shrinkage): {n_singleton:,}')
print(f'\nSample — 5 unique keys showing LOO stats:')
print(
    df_train_enriched
    [['degree_course_key', 'support_count', 'course_pass_rate',
      'course_avg_mark', 'course_retake_rate', 'course_difficulty_fallback_level']]
    .drop_duplicates('degree_course_key')
    .head(5)
    .to_string(index=False)
)

LOO enrichment complete: 450,465 train rows
Singleton rows (n_loo == 0, collapse to global avg via shrinkage): 141

Sample — 5 unique keys showing LOO stats:
degree_course_key  support_count  course_pass_rate  course_avg_mark  course_retake_rate  course_difficulty_fallback_level
  3.111__1016.111            147          0.951056        72.243365            0.020408                                 1
  3.111__1017.111             76          0.904441         71.84002            0.052632                                 1
  3.111__1019.111            292          0.912905        72.800134            0.089041                                 1
   3.111__431.111            950          0.803945        64.660456            0.240000                                 1
   3.111__432.111            887          0.837736        68.650101            0.211950                                 1


**What this shows:** `support_count` for train rows is `n_loo` (the course's total count minus 1), not the full count — this is expected. Singleton rows (support_count = 0) receive the global average via shrinkage, which is the correct conservative behaviour. All train rows get `course_difficulty_fallback_level = 1` since every train row's key exists in the training aggregation.

**Step B.13** — Apply the 3-level fallback chain to `df_valid` and `df_test`. Print the fallback distribution (counts and %) for all three splits.

In [9]:
# Pre-build Series for O(n) vectorised .map() lookup at each level
_l1_pr = key_stats.set_index('degree_course_key')['course_pass_rate']
_l1_am = key_stats.set_index('degree_course_key')['course_avg_mark']
_l1_rr = key_stats.set_index('degree_course_key')['course_retake_rate']
_l1_sc = key_stats.set_index('degree_course_key')['support_count']

_l2_pr = cid_stats.set_index('course_id')['course_pass_rate']
_l2_am = cid_stats.set_index('course_id')['course_avg_mark']
_l2_rr = cid_stats.set_index('course_id')['course_retake_rate']
_l2_sc = cid_stats.set_index('course_id')['support_count']


def _enrich_split(df_split, split_name):
    """Apply 3-level fallback to a val or test split. Returns enriched copy."""
    dck = df_split['degree_course_key']
    cid = dck.str.rsplit('__', n=1).str[-1]

    pr_l1 = dck.map(_l1_pr);  am_l1 = dck.map(_l1_am)
    rr_l1 = dck.map(_l1_rr);  sc_l1 = dck.map(_l1_sc)
    l1_hit = pr_l1.notna()

    pr_l2 = cid.map(_l2_pr);  am_l2 = cid.map(_l2_am)
    rr_l2 = cid.map(_l2_rr);  sc_l2 = cid.map(_l2_sc)
    l2_hit = (~l1_hit) & pr_l2.notna()

    out = df_split.copy()
    out['course_pass_rate']   = np.where(l1_hit.values, pr_l1.values,
                                np.where(l2_hit.values, pr_l2.values,
                                         global_vals['course_pass_rate']))
    out['course_avg_mark']    = np.where(l1_hit.values, am_l1.values,
                                np.where(l2_hit.values, am_l2.values,
                                         global_vals['course_avg_mark']))
    out['course_retake_rate'] = np.where(l1_hit.values, rr_l1.values,
                                np.where(l2_hit.values, rr_l2.values,
                                         global_vals['course_retake_rate']))
    out['support_count']      = np.where(l1_hit.values, sc_l1.values,
                                np.where(l2_hit.values, sc_l2.values, 0)).astype(int)
    out['course_difficulty_fallback_level'] = np.where(l1_hit.values, 1,
                                              np.where(l2_hit.values, 2, 3)).astype(int)

    n = len(out)
    fb_counts = pd.Series(out['course_difficulty_fallback_level'].values).value_counts().sort_index()
    print(f'\n{split_name} ({n:,} rows) — fallback distribution:')
    for lvl, cnt in fb_counts.items():
        print(f'  Level {int(lvl)}: {cnt:>8,} rows  ({cnt / n * 100:.2f}%)')
    return out


df_valid_enriched = _enrich_split(df_valid, 'valid')
df_test_enriched  = _enrich_split(df_test,  'test')

# Train: all level 1 by construction (LOO uses train-only keys)
n_tr = len(df_train_enriched)
print(f'\ntrain ({n_tr:,} rows) — fallback distribution:')
tr_fb = df_train_enriched['course_difficulty_fallback_level'].value_counts().sort_index()
for lvl, cnt in tr_fb.items():
    print(f'  Level {int(lvl)}: {cnt:>8,} rows  ({cnt / n_tr * 100:.2f}%)')


valid (156,097 rows) — fallback distribution:
  Level 1:  120,858 rows  (77.42%)
  Level 2:    9,612 rows  (6.16%)
  Level 3:   25,627 rows  (16.42%)

test (110,008 rows) — fallback distribution:
  Level 1:   49,669 rows  (45.15%)
  Level 2:   25,640 rows  (23.31%)
  Level 3:   34,699 rows  (31.54%)

train (450,465 rows) — fallback distribution:
  Level 1:  450,465 rows  (100.00%)


**What this shows:** How often each fallback level is used in val and test. Level 1 (exact match) is the ideal case. A high level-2 or level-3 percentage in val/test means many courses in those splits were never seen in training by their full `degree_course_key` — worth noting if it's unexpectedly high. Train is 100% level 1 by definition (every train row's key exists in the LOO aggregation).

**Step B.14** — Sanity check: pick 10 `degree_course_key` values present in all three splits and confirm that `df_valid_enriched` and `df_test_enriched` show identical difficulty values for each (same train-lookup). Also show the `key_stats` reference values for comparison.

In [10]:
keys_tr = set(df_train_enriched['degree_course_key'].unique())
keys_vl = set(df_valid_enriched['degree_course_key'].unique())
keys_ts = set(df_test_enriched['degree_course_key'].unique())
keys_in_all = keys_tr & keys_vl & keys_ts

print(f'Keys present in all three splits: {len(keys_in_all):,}')
sample_keys = sorted(keys_in_all)[:10]

SHOW = ['degree_course_key', 'support_count',
        'course_pass_rate', 'course_avg_mark',
        'course_difficulty_fallback_level']

pd.set_option('display.float_format', '{:.4f}'.format)

# Val and test must show identical values for each key (both use the same key_stats lookup)
for split_name, df_e in [('valid', df_valid_enriched), ('test', df_test_enriched)]:
    print(f'\n--- {split_name} ---')
    subset = (
        df_e.loc[df_e['degree_course_key'].isin(sample_keys)]
        .drop_duplicates('degree_course_key')
        [SHOW]
        .sort_values('degree_course_key')
    )
    print(subset.to_string(index=False))

# key_stats reference (non-LOO) for comparison — train row values will be close but not identical
print('\n--- key_stats reference (non-LOO; train per-row values differ slightly by design) ---')
ref = (
    key_stats.loc[key_stats['degree_course_key'].isin(sample_keys)]
    [['degree_course_key', 'support_count', 'course_pass_rate', 'course_avg_mark']]
    .sort_values('degree_course_key')
    .assign(course_difficulty_fallback_level=1)
)
print(ref.to_string(index=False))

pd.reset_option('display.float_format')

Keys present in all three splits: 700

--- valid ---
degree_course_key  support_count  course_pass_rate  course_avg_mark  course_difficulty_fallback_level
  1.111__1015.111            192            0.9001          75.1257                                 1
  1.111__1016.111            914            0.9720          79.5135                                 1
  1.111__1017.111            299            0.9211          76.6729                                 1
  1.111__1018.111            524            0.8839          66.7107                                 1
  1.111__1019.111            938            0.9330          78.3190                                 1
  1.111__1020.111           1119            0.9498          76.5739                                 1
  1.111__1021.111           1517            0.8802          69.0629                                 1
  1.111__1032.111           1044            0.8814          65.4480                                 1
  1.111__1033.111           1

**What this shows:** For any given `degree_course_key`, the val and test rows should show **identical** `course_pass_rate`, `course_avg_mark`, and `support_count` (since they both look up from the same `key_stats` table). The `key_stats` reference row shows the non-LOO value, which is very close but not exactly equal to what individual train rows see (by design — the LOO version excludes each row's own contribution).

**Save** — Overwrite `SPLIT_DATA_DIR/df_train/valid/test.parquet` with the enriched versions. Assert the column count increased by exactly 5 versus what was loaded in Step A.1.

In [11]:
import os

# SPLIT_DATA_DIR defined in Step A.0 — reuse it here, no separate path to keep in sync
TRAIN_PATH = os.path.join(SPLIT_DATA_DIR, 'df_train.parquet')
VALID_PATH = os.path.join(SPLIT_DATA_DIR, 'df_valid.parquet')
TEST_PATH  = os.path.join(SPLIT_DATA_DIR, 'df_test.parquet')

# Column count before (as loaded from disk in Step A.1) vs after enrichment
cols_before = df_train.shape[1]
cols_after  = df_train_enriched.shape[1]
n_new_cols  = cols_after - cols_before

print(f'Column count loaded from disk (Step A.1) : {cols_before}')
print(f'Column count after enrichment            : {cols_after}')
print(f'New columns added                        : {n_new_cols}  (expected 5)')

assert n_new_cols == 5, (
    f'Expected exactly 5 new columns, got {n_new_cols}. '
    f'Check for duplicate or missing difficulty columns.'
)

# Overwrite the basic splits with enriched versions
df_train_enriched.to_parquet(TRAIN_PATH, index=True)
df_valid_enriched.to_parquet(VALID_PATH, index=True)
df_test_enriched.to_parquet(TEST_PATH,   index=True)

NEW_COLS = ['course_pass_rate', 'course_avg_mark', 'course_retake_rate',
            'course_difficulty_fallback_level', 'support_count']
print(f'\nNew columns: {NEW_COLS}')
print()

for label, path, df_e in [
    ('train', TRAIN_PATH, df_train_enriched),
    ('valid', VALID_PATH, df_valid_enriched),
    ('test',  TEST_PATH,  df_test_enriched),
]:
    size_mb = os.path.getsize(path) / 1_048_576
    print(f'  {label:<6}: {df_e.shape}  ->  {path}  ({size_mb:.1f} MB)')

Column count loaded from disk (Step A.1) : 61
Column count after enrichment            : 66
New columns added                        : 5  (expected 5)

New columns: ['course_pass_rate', 'course_avg_mark', 'course_retake_rate', 'course_difficulty_fallback_level', 'support_count']

  train : (450465, 66)  ->  D:/AI/Real projects/Academic_Advisor/data/model_data\df_train.parquet  (18.4 MB)
  valid : (156097, 66)  ->  D:/AI/Real projects/Academic_Advisor/data/model_data\df_valid.parquet  (6.2 MB)
  test  : (110008, 66)  ->  D:/AI/Real projects/Academic_Advisor/data/model_data\df_test.parquet  (4.3 MB)


**What this shows:** The before/after column count confirms exactly 5 difficulty columns were added. The assertion fails loudly if there's a mismatch (e.g., a column was accidentally duplicated or omitted). After this cell runs, the files in `SPLIT_DATA_DIR` are the **final enriched splits** — the basic versions from `split_diagnostics.ipynb` have been overwritten and downstream model notebooks should load from here.